# Day 4.3: MLOps Feature Store, Feature Consistency, and Monitoring

In Day 4.1 and Day 4.2, we trained, tracked, registered, and served a taxi availability prediction model.

This notebook focuses on the next practical MLOps question:

```text
When a model is served in an application, how do we keep features, model versions, and monitoring consistent?
```

We use the same taxi/weather project and introduce four ideas from the Day 4 lecture:

| Topic | Question |
|---|---|
| Feature Store | Where should shared feature definitions live? |
| Training-serving skew | Are training features and serving features calculated the same way? |
| Canary deployment | How can we test a new model version without switching everyone at once? |
| Data drift / concept drift | How do we know the world has changed after deployment? |

The workflow stays simple:

```text
data -> feature store -> train -> track -> register -> serve -> log -> monitor
```

## 0. Feast Workflow Map

Before looking at individual files, build the workflow picture first. A feature store is not just another training script. It connects the offline training path and the online serving path through the same named feature contract.

The flow in this demo is:

```text
prepared taxi/weather dataset
-> offline feature table
-> Feast entity + feature views + feature services
-> apply definitions to the registry
-> materialize latest values to the online store
-> train with historical feature retrieval
-> serve with online feature retrieval
```

The key objects are created in this order:

| Workflow step | Feast idea | File to inspect |
|---|---|---|
| Prepare historical feature values | Offline feature table | `feature_store_demo/build_feature_data.py` |
| Configure the local feature store | Registry and online store | `feature_store_demo/feature_store.yaml` |
| Define the feature contract | Entity, FeatureView, FeatureService | `feature_store_demo/feature_repo.py` |
| Register and load features for serving | `apply` and `materialize` | `feature_store_demo/setup_feature_store.py` |
| Train using historical features | `get_historical_features(...)` | `feature_store_demo/train_with_feast.py` |
| Predict using online features | `get_online_features(...)` | `feature_store_demo/online_predict_with_feast.py` |
| Use the same idea in an API | Feature retrieval inside serving | `feature_store_demo/dashboard_app.py` |

The important handoff is:

```text
feature_repo.py defines the contract
train_with_feast.py asks Feast for historical features from that contract
dashboard_app.py asks Feast for online features from that contract
MLflow records which model version used which feature contract
```

So the goal is not to memorize every file. The goal is to see how one feature definition can support both training and serving.



## 0. The Problem Before The Full Workflow

Before we connect the full MLflow workflow, pause on one practical question:

```text
If the app loads models:/taxi_availability_2h_model@champion,
how does the app know which features to prepare?
```

In Day 4.1 and Day 4.2, MLflow helped us record the model run and load the champion model. We also logged metadata such as:

```text
data_version = v1
feature_version = v1
features = hour,day_of_week,is_weekend,is_rain_forecast_2h,weather_category_id,current_taxi_count
```

This is useful, but it is still not a complete system.

Discussion:

> Is `feature_version=v1` enough to guarantee the dashboard calculates the same features as the training script?



## 0.1 Same Model Name, Different Feature Needs

A registered model name is a business name, not necessarily a feature contract.

For example, both of these can be versions of the same registered model:

```text
taxi_availability_2h_model version 1
taxi_availability_2h_model version 2
```

Version 1 might use:

```text
hour, day_of_week, is_weekend
```

Version 2 might use:

```text
hour, day_of_week, is_weekend, is_rain_forecast_2h, weather_category_id, current_taxi_count
```

The dashboard may still load the same URI:

```text
models:/taxi_availability_2h_model@champion
```

So the model pointer can change without the serving code clearly knowing which feature logic must change with it.


## 0.2 Why Feature Version Alone Is Not Enough

`feature_version` is good metadata, but metadata is not the same as executable feature logic.

The risky pattern is duplicated feature engineering code:

| Place | What it does |
|---|---|
| Training script | Reads historical CSV and creates model input features |
| Dashboard / API | Reads live data and creates model input features again |

Even if both sides say `feature_version=v1`, small differences can happen:

```text
Different timestamp handling
Different weather category mapping
Different missing value default
Different feature order
One new feature added in training but not serving
```

This is called training-serving skew.

It means the model was trained with one meaning of the data, but served with a slightly different meaning of the data.


## 0.3 Feature Store: The Missing Layer

A feature store gives the team a shared place to define, store, and retrieve features.

The core idea:

```text
Training and online inference should ask for the same feature service.
```

A feature store usually has:

| Concept | Meaning in our taxi project |
|---|---|
| Entity | `planning_area` |
| Offline store | Historical features for training |
| Online store | Latest features for prediction |
| Feature view | A versioned group of features, such as `taxi_area_features_v1` |
| Feature service | The exact feature set required by a model, such as `taxi_availability_model_v1` |

Now the model metadata can point to a real contract:

```text
feature_store = feast
feature_service = taxi_availability_model_v1
feature_view = taxi_area_features_v1
entity = planning_area
```

This is more systematic than only writing `feature_version=v1`.


## 0.3.1 Why This Matters Online

In Day 4.2, the serving API creates features inside `dashboard_app.py` before calling the model:

```text
latest taxi/weather data
-> manually build feature columns
-> model.predict(...)
```

That is easy to understand, but it creates a risk: training code and serving code may slowly become different.

A feature store changes the online path to:

```text
request area
-> ask Feast for the named feature service
-> receive feature columns in the expected order
-> load MLflow champion model
-> model.predict(...)
```

So Day 4.2 teaches **how online inference works**. Day 4.3 teaches **how teams keep online inference consistent with training**.



## 0.4 What We Will Build With Feast

We will build a small local Feast feature store for the taxi availability model.

The goal is not to build a production feature platform today. The goal is to make the missing layer visible:

```text
same entity
same feature definitions
same feature service name
training path and online path both retrieve from Feast
```

In the unified Day 4 folder, the Feast files live here:

```text
day_4/day_4_mlflow/feature_store_demo/
```

In `feature_store_demo/feature_repo.py`, we define two comparable feature versions:

| Feature version | Feature view | Feature service | Features |
|---|---|---|---|
| `v0` | `taxi_area_features_v0` | `taxi_availability_model_v0` | time features only |
| `v1` | `taxi_area_features_v1` | `taxi_availability_model_v1` | time + weather + current taxi count |

This mirrors the MLflow workflow, but now the feature versions are not only run parameters. They are named Feast objects.



## 0.5 Step 1 - Prepare A Feature Data Source

Feast needs an offline source of historical feature values.

For this local demo, the source is a Parquet file created from the existing Day 4 taxi/weather training CSV.

Conceptually, we create two datasets:

| Dataset | Purpose |
|---|---|
| Feature table | historical feature values for each `planning_area` at each event time |
| Entity table | the planning area and timestamp for each training example, plus the target label |

Why separate them?

```text
Feature table = what was known about an entity at a time.
Entity table = which entity-time examples we want to train on.
```

This separation is important. During training, the model should ask:

```text
For this planning area and this timestamp, what feature values were available then?
```

That is the feature-store way of thinking.


## 0.6 Step 2 - Configure The Feast Repository

The Feast repository starts with `feature_store.yaml`:

```yaml
project: taxi_availability_feature_store
provider: local
registry: data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
entity_key_serialization_version: 3
```

Meaning:

| Config | Meaning |
|---|---|
| `project` | namespace for this feature store |
| `provider: local` | use local development mode |
| `registry` | where Feast records feature definitions |
| `online_store` | where Feast stores latest feature values for serving |

For class, local Parquet + SQLite is enough.

Production teams may use BigQuery, Snowflake, Redshift, Spark, DynamoDB, Redis, or other stores, but the concepts are the same.


## 0.7 Step 3 - Define The Entity

Open `feature_repo.py`.

The first object is the entity:

```python
planning_area = Entity(
    name="planning_area",
    join_keys=["planning_area"],
    value_type=ValueType.STRING,
)
```

An entity is the key we use to retrieve features.

For our taxi model:

```text
One prediction is for one Singapore planning area.
```

So `planning_area` is our entity.

When we ask Feast for online features, we will pass:

```python
{"planning_area": "TAMPINES"}
```


## 0.8 Step 4 - Define The Data Source

Next, we tell Feast where the historical feature values live:

```python
taxi_area_feature_source = FileSource(
    name="taxi_area_feature_source",
    path="data/taxi_area_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)
```

Important fields:

| Field | Meaning |
|---|---|
| `path` | local Parquet file containing historical features |
| `timestamp_field` | event time of the feature value |
| `created_timestamp_column` | when this record was created |

### Point-in-time correct

Point-in-time correct means:

```text
When training on an example from time T, only use feature values that would have been available at time T.
```

For example, suppose we train one row for `ANG MO KIO` at 10:00. The label is taxi count around 12:00, but the input features should still come from what we knew at 10:00:

```text
Allowed: taxi count at 10:00, weather forecast available at 10:00
Not allowed: taxi count at 12:00, weather update created after 10:00
```

If training accidentally uses future information, the model may look very good offline but fail in real serving. This is called data leakage.

Feast uses timestamp fields to help retrieve the right historical feature values for each entity-time training row.



## 0.9 Step 5 - Define Feature Views v0 And v1

A feature view is a named group of features that come from the same source.

In this demo, `v0` is the simple baseline:

```python
taxi_area_features_v0 = FeatureView(
    name="taxi_area_features_v0",
    entities=[planning_area],
    schema=[
        Field(name="hour", dtype=Int64),
        Field(name="day_of_week", dtype=Int64),
        Field(name="is_weekend", dtype=Int64),
    ],
    source=taxi_area_feature_source,
    online=True,
)
```

`v1` adds richer context:

```python
taxi_area_features_v1 = FeatureView(
    name="taxi_area_features_v1",
    entities=[planning_area],
    schema=[
        Field(name="hour", dtype=Int64),
        Field(name="day_of_week", dtype=Int64),
        Field(name="is_weekend", dtype=Int64),
        Field(name="is_rain_forecast_2h", dtype=Int64),
        Field(name="weather_category_id", dtype=Int64),
        Field(name="current_taxi_count", dtype=Int64),
    ],
    source=taxi_area_feature_source,
    online=True,
)
```

Now the feature version is a real object in the feature store, not just a string in a training script.


## 0.10 Step 6 - Define Feature Services v0 And v1

A feature service is the model-facing feature contract.

In this demo:

```python
taxi_availability_model_v0 = FeatureService(
    name="taxi_availability_model_v0",
    features=[taxi_area_features_v0],
)
```

```python
taxi_availability_model_v1 = FeatureService(
    name="taxi_availability_model_v1",
    features=[taxi_area_features_v1],
)
```

This is the object that training and online inference both request.

Main idea:

```text
The model should not only say feature_version=v1.
The model should point to feature_service=taxi_availability_model_v1.
```

That feature service tells us exactly which features the model expects.


## 0.11 Step 7 - Apply Definitions And Materialize Online Features

After defining entities, feature views, and feature services, we apply them to Feast.

Two things happen:

| Step | What happens |
|---|---|
| Apply definitions | Feast records the entity, feature views, and feature services in its registry |
| Materialize features | Feast copies the latest feature values into the online store |

After this, Feast has two sides:

```text
offline historical features for training
online latest features for serving
```

For this local snapshot, the setup script materializes the whole available time range from the Parquet file. This avoids a common demo problem: if the source data is a few weeks old, a short TTL can make online features appear as `None`.



## 0.12 Step 8 - Train With v0 And v1 From Feast

Now the training script does not manually choose every feature column from scratch.

Instead, it asks Feast for a named feature service:

```python
feature_service = store.get_feature_service(feature_service_name)
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service,
).to_df()
```

The important change is conceptual:

```text
Before: training script owns the feature logic.
After: training script asks the feature store for the model's feature contract.
```

The run can still be logged to MLflow. Now the MLflow record can include both model information and feature-store information:

```text
feature_store = feast
feature_version = v0 or v1
feature_service = taxi_availability_model_v0 or taxi_availability_model_v1
feature_view = taxi_area_features_v0 or taxi_area_features_v1
entity = planning_area
```


## 0.13 Step 9 - Fetch Online Features With v0 And v1

After materialization, serving can fetch online features for one planning area.

The useful comparison is:

```text
v0 returns: hour, day_of_week, is_weekend
v1 returns: hour, day_of_week, is_weekend, is_rain_forecast_2h, weather_category_id, current_taxi_count
```

This answers the earlier question:

```text
How does serving know which features to fetch?
```

It uses the same feature service version that the model was trained with.

A useful debugging habit is to first fetch features without loading the model. If the feature values are missing or wrong, fix Feast before blaming the model.


## 0.14 Step 10 - Use Feast Inside The Dashboard

The command-line script proves that Feast online retrieval works. The same idea becomes clearer when it appears inside the dashboard.

The dashboard changes only one important part:

```text
Day 4.2: FastAPI app builds feature columns manually.
Day 4.3: FastAPI app asks Feast online store for the configured feature service.
```

Inside the dashboard, the serving flow becomes:

```text
Dashboard receives request
-> dashboard asks Feast for online features
-> dashboard loads MLflow champion model
-> model predicts using the Feast feature vector
-> prediction log records model + feature service
```

In the response JSON, look for fields such as:

```text
feature_source = feast_online_store
feature_service = taxi_availability_model_v1
feature_view = taxi_area_features_v1
features = {...}
```

This makes the feature contract visible at serving time.



## 0.15 How Feast And MLflow Work Together

Feature store and MLflow solve different parts of the MLOps problem.

| Tool | Main responsibility |
|---|---|
| Feast | define and retrieve features consistently |
| MLflow Tracking | record training runs, params, metrics, datasets, and artifacts |
| MLflow Registry | manage model versions and aliases such as `champion` |

A good production record should connect both:

```text
Registered model: taxi_availability_2h_model version 3
Alias: champion
Feature store: feast
Feature service: taxi_availability_model_v1
Feature view: taxi_area_features_v1
Entity: planning_area
```

Now when the champion model changes, we can also see which feature contract it expects.

After this point, the rest of the notebook returns to the MLflow lifecycle:

```text
track -> evaluate -> register -> load -> serve -> monitor
```


## 0.16 Deployment Pattern: Champion And Canary

In Day 4.2, the serving app loads the current production model through an MLflow alias:

```text
models:/taxi_availability_2h_model@champion
```

That is convenient because the app does not need to hard-code a model version number.

In real systems, teams often avoid switching all traffic to a new model immediately. A safer pattern is canary deployment:

| Alias / version | Meaning |
|---|---|
| `champion` | current main production model |
| `candidate` or `challenger` | new model being tested |
| canary traffic | a small share of requests routed to the new model first |

In this local demo, we usually switch the `champion` alias manually and reload the model cache. The concept is the same: the serving layer should make model version changes visible and reversible.



## 0.17 Training-Serving Skew

Training-serving skew means the model sees one kind of feature during training and a slightly different kind of feature during online prediction.

Common causes:

| Cause | Example in this project |
|---|---|
| Different feature code | training creates `is_weekend` one way, dashboard creates it another way |
| Different feature order | model expects weather columns after time columns, serving sends them in another order |
| Different missing-value logic | training fills missing weather with 0, serving fills it with -1 |
| Different timestamp logic | training uses historical event time, serving uses current system time incorrectly |

A feature store reduces this risk because training and serving request a named feature service instead of copying feature logic in multiple places.


## 0.18 Data Drift And Concept Drift

After deployment, monitoring is not only about whether the API is alive. We also care whether the model is still seeing the kind of world it was trained for.

| Concept | Meaning | Taxi/weather example |
|---|---|---|
| Data drift | input distribution changes | many more rainy hours than the training data had |
| Concept drift | relationship between inputs and target changes | rain used to reduce taxi availability, but new demand patterns change the relationship |
| Training-serving skew | training and serving calculate features differently | dashboard creates a different `weather_category_id` mapping than training |

These are different problems. A useful monitoring log records the model URI, feature service, input features, prediction, and later the actual outcome when it becomes available.


## 0.19 Before Retraining

When model quality drops, retraining is not always the first action. A production team should first ask:

| Check | Why it matters |
|---|---|
| Data pipeline health | bad input data can make a good model look bad |
| Feature consistency | skew can hurt predictions even if the model is unchanged |
| Label quality | delayed or wrong labels can create false alarms |
| Metric definition | the business metric may not match the training metric |
| Model comparison | a new model should beat the current champion on relevant slices |
| Rollback plan | deployment should be reversible if the new version fails |

The important habit is to connect model behavior back to data, features, and deployment records instead of treating the model file as the whole system.



## 0.20 Classroom Practice: Classify The Problem

For each case, decide whether it is mainly data drift, concept drift, or training-serving skew.

| Scenario | Likely category |
|---|---|
| The dashboard accidentally sends columns in the wrong order | training-serving skew |
| A new public holiday pattern changes ride demand | concept drift |
| Weather data suddenly contains a new category not seen before | data drift |
| Training uses UTC time but serving uses local Singapore time without conversion | training-serving skew |
| Rainy days become much more frequent than in the training period | data drift |

This is a good discussion point before moving back into MLflow tracking, registry, and serving.


## 1. How This Connects Back To MLflow

Feature Store solves the feature consistency problem. MLflow still solves the model lifecycle problem.

Here, we use MLflow for these practical questions:

| MLflow area | Practical question |
|---|---|
| Tracking | What happened during training? |
| Dataset / parameter logging | Which data and feature version did this run use? |
| Models | What files and dependencies are saved with the model? |
| Evaluation | How good is the model on test data? |
| Registry | Which model version should the app use? |
| Serving | How does an API load and call the model? |
| Monitoring logs | What happened after deployment? |

The full MLOps story is not one tool. It is the connection between data, features, model versions, serving, and monitoring.



## 2. Tracking: What Happened During Training?

In our project, tracking means recording:

```text
data_version
feature_version
features
feature_count
model_type
MAE / RMSE / R2
model artifact
```

After a training run, open MLflow and inspect the run page:

```text
http://localhost:5000
```

Discussion:

> Can you reconstruct what this model used just by reading the MLflow run page?



## 3. Dataset Tracking: Which Data Did We Use?

The MLflow docs describe dataset tracking as a way to record dataset source, schema, profile, and lineage.

Here, we keep the idea simple:

```text
data_version v0 = first training CSV
data_version v1 = newer 0606 training CSV
data_version v3 = future collected data for another experiment
```

The comparison question is:

> Did using newer data change MAE, RMSE, or R2?

This is the beginning of dataset-aware model development. We are not only asking which algorithm is better; we are also asking which data produced the model.



## 4. MLflow Models: What Is Actually Saved?

MLflow models are saved in a standard format. In our artifact folder, you should see files such as:

```text
model/MLmodel
model/model.pkl
model/requirements.txt
model/python_env.yaml
```

The important idea:

> A model is not just a Python object. It also needs metadata and dependency information so another program can load it later.

In our scripts, we load models through the generic pyfunc interface:

```python
model = mlflow.pyfunc.load_model("models:/taxi_availability_2h_model@champion")
prediction = model.predict(input_dataframe)
```



## 5. Evaluation: How Good Is The Model?

The MLflow docs include `mlflow.models.evaluate()` for automated evaluation. For our first regression project, we keep evaluation explicit and readable:

```text
MAE  = average absolute error, in number of taxis
RMSE = error metric that penalizes big mistakes more
R2   = how much better we are than predicting a simple average
```

Discussion:

> Which metric is easiest to explain to a non-technical stakeholder?

Usually MAE is easiest because the unit is still number of taxis.



## 6. Registry: Which Model Should We Use?

The Model Registry gives the application a stable model name:

```text
taxi_availability_2h_model
```

The `champion` alias points to the model version currently selected for serving:

```text
models:/taxi_availability_2h_model@champion
```

This avoids hard-coding a model version number in the application.

The serving app can stay stable while the team updates which model version is marked as `champion`.


## 7. Serving: Use The Model From A Web App

The MLflow serving docs explain local serving and REST APIs. Here, we show two practical serving styles:

1. a focused FastAPI model API in `day_4_mlflow/serve_model.py`;
2. the Day 1 taxi/weather dashboard with a new prediction feature.

The Day 1 dashboard now has an endpoint:

```text
GET /api/model/predict-2h?area=ANG%20MO%20KIO
```

It loads the registered MLflow model and predicts the taxi count about 2 hours later for the selected area.



## 8. Run The Dashboard With Prediction

There are two dashboard versions in the unified Day 4 Docker folder:

| Folder | Main idea |
|---|---|
| `day_4/day_4_mlflow/dashboard_app.py` | model serving with manually prepared feature columns |
| `day_4/day_4_mlflow/feature_store_demo/dashboard_app.py` | model serving with features retrieved from Feast |

Compare the two serving approaches:

```text
Day 4.2: serving code prepares feature columns itself
Day 4.3: serving code asks Feast for the model's feature service
```

The second approach is closer to production MLOps because the model's feature contract is explicit and reusable.


## 9. Monitoring: What Happened After Deployment?

Monitoring starts with simple records:

```text
timestamp
input features
prediction
model URI or model version
```

In Day 4.2, `serve_model.py` logs prediction requests to:

```text
day_4_mlflow/prediction_logs.db
```

This is not a full monitoring system. It is the first habit: keep enough records so we can debug and compare model behavior later.


## 10. Example: Simple MLP

The MLflow deep learning docs show that the same lifecycle applies to neural networks.

For a lightweight exercise, we use scikit-learn's `MLPRegressor` and compare it against the linear model.

Questions:

1. Did the MLP improve MAE, RMSE, or R2?
2. What new parameters appeared in MLflow?
3. Would you register the MLP as champion? Why or why not?

The important idea is not that a neural network is automatically better. The important idea is that different model types can go through the same MLOps workflow: track, evaluate, register, serve, and monitor.



## 11. Complete Command Line: Run The Demo From Zero

Use this section to run the full Day 4.3 demo from zero. The earlier sections explain the concepts; this section gives the command sequence.

### 1. Start the unified Day 4 container

```powershell
cd day_4\day_4_mlflow
docker compose up --build
```

Keep this terminal open. MLflow is available at:

```text
http://localhost:5000
```

### 2. Build and materialize Feast features

Open a second terminal:

```powershell
docker exec -it -w /workspace/feature_store_demo day4-mlops python build_feature_data.py
docker exec -it -w /workspace/feature_store_demo day4-mlops python setup_feature_store.py
```

### 3. Check online features before using the model

```powershell
docker exec -it -w /workspace/feature_store_demo day4-mlops python online_predict_with_feast.py --planning-area TAMPINES --feature-version v1 --no-model
```

`--no-model` means: fetch Feast online features only. This is useful for debugging before model prediction.

### 4. Train and register a model that uses Feast

```powershell
docker exec -it -w /workspace/feature_store_demo day4-mlops python train_with_feast.py --feature-version v1 --tracking-uri http://127.0.0.1:5000 --register-model --model-name taxi_availability_2h_model --model-alias champion
```

### 5. Start the Feast dashboard

Open a third terminal:

```powershell
docker exec -it -w /workspace/feature_store_demo day4-mlops bash
```

Inside the container:

```bash
cp /workspace/live_dashboard_data.db /tmp/live_dashboard_data.db
DAY_4_DASHBOARD_DB_PATH=/tmp/live_dashboard_data.db \
MLFLOW_TRACKING_URI=http://127.0.0.1:5000 \
MLFLOW_MODEL_URI=models:/taxi_availability_2h_model@champion \
FEAST_FEATURE_SERVICE=taxi_availability_model_v1 \
python -m uvicorn dashboard_app:app --host 0.0.0.0 --port 8001
```

Open:

```text
http://localhost:8001/predict-dashboard
```

### 6. Stop the environment

When finished:

```powershell
cd day_4\day_4_mlflow
docker compose down
```



## 12. Extension Design Question: Real-Time Features With Feast

This local demo uses a materialized snapshot:

```text
prepared CSV
-> build_feature_data.py
-> taxi_area_features.parquet
-> setup_feature_store.py materialize
-> online_store.db
-> dashboard reads online features
```

That is enough to understand the feature-store workflow, but it is not truly real-time.

Design question:

> If we wanted the Feast dashboard to make real-time taxi predictions, what would need to change?

Hints:

1. **Feature generation still needs a pipeline**

   Feast does not automatically calculate features from raw APIs. Something upstream must continuously read taxi/weather data and compute feature values.

   ```text
   taxi/weather APIs
   -> feature pipeline
   -> latest feature values
   -> Feast online store
   ```

2. **The online store must be updated continuously**

   Instead of materializing a fixed Parquet snapshot once, a scheduler or streaming job would keep writing the latest feature values into the online store.

   Possible tools:

   ```text
   Airflow batch job
   Kafka + Flink streaming job
   Spark structured streaming
   Python scheduler
   cloud data pipeline
   ```

3. **The dashboard should only retrieve features**

   The serving API should stay simple:

   ```text
   request area
   -> Feast get_online_features(...)
   -> MLflow champion model
   -> prediction
   -> prediction log
   ```

4. **Freshness becomes a monitoring problem**

   Real-time serving must track whether features are fresh enough:

   ```text
   latest feature timestamp
   feature age in minutes
   missing feature rate
   online store write failures
   prediction latency
   ```

5. **Offline and online logic must stay consistent**

   The same feature definitions should support training and serving. If the real-time pipeline changes how `weather_category_id` is calculated, the training pipeline must not silently use a different rule.

A good answer should mention both sides:

```text
Feature pipeline keeps the online store fresh.
Feast provides consistent retrieval for training and serving.
```



## 13. Final Workflow

By the end of Day 4, you should be able to explain the MLOps workflow in plain language:

```text
Collect data
-> Prepare data and features
-> Define a feature contract
-> Train model
-> Track experiment
-> Evaluate model
-> Register model
-> Assign champion alias
-> Load model in an app
-> Serve prediction
-> Log requests and predictions
-> Monitor drift, skew, and model quality
-> Retrain or roll back when needed
```

The key idea is not that MLflow or Feast magically makes a model good. The key idea is that production ML needs visible records of data, features, model versions, serving behavior, and monitoring signals.


